# 17장 워드클라우드와 텍스트 시각화

이 장은 PDF 교재의 NLP, 텍스트 처리, LLM 응답 분석 흐름을 바탕으로 텍스트 데이터를 시각화하는 방법을 정리합니다.

LLM 실습에서는 질문, 답변, 문서 조각, 검색 결과, 채팅 기록 같은 텍스트가 계속 만들어집니다. 이런 텍스트를 그대로 읽는 것도 중요하지만, 단어 빈도와 주요 키워드를 시각화하면 전체 흐름을 더 빠르게 파악할 수 있습니다.

## 이 장에서 다루는 내용

1. 텍스트 시각화가 필요한 이유
2. 워드클라우드 개념
3. 텍스트 전처리 기본
4. 단어 빈도 계산
5. 막대그래프로 단어 빈도 시각화
6. WordCloud로 워드클라우드 만들기
7. 한국어 텍스트 시각화 주의점
8. LLM 답변과 RAG 문서 분석
9. Gradio로 텍스트 시각화 UI 만들기
10. 오류 해결과 확장 방향

이 장은 `LLM/llm.ipynb`와 `LLM/llm2.ipynb`에서 생성되는 텍스트 답변, 문서 검색 결과, DB 질의 결과 설명을 분석하는 보조 실습입니다.


## 17.1 텍스트 시각화가 필요한 이유

텍스트 시각화는 많은 문장을 한눈에 이해하기 위해 사용합니다.

LLM 실습에서 텍스트 시각화가 유용한 상황은 다음과 같습니다.

- 여러 LLM 답변에서 자주 등장하는 키워드를 확인할 때
- RAG 검색 결과가 어떤 주제에 집중되어 있는지 볼 때
- 채팅 기록에서 사용자가 많이 묻는 주제를 파악할 때
- DB 챗봇 결과 설명에 어떤 용어가 반복되는지 확인할 때
- 자기소개서, 리뷰, 설문 응답 같은 긴 텍스트를 빠르게 요약할 때

워드클라우드는 중요한 단어를 크게 보여주는 방식이고, 막대그래프는 단어 빈도를 정확하게 비교하는 방식입니다. 두 방법을 함께 사용하면 직관성과 정확성을 모두 얻을 수 있습니다.


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 설치 준비
# 이 셀은 텍스트 시각화 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# wordcloud는 단어 빈도를 바탕으로 워드클라우드 이미지를 생성하는 라이브러리입니다.
!pip install wordcloud

# matplotlib은 그래프와 이미지를 그릴 때 사용하는 대표적인 시각화 라이브러리입니다.
!pip install matplotlib

# pandas는 텍스트 빈도 결과를 표 형태로 다룰 때 사용합니다.
!pip install pandas

# konlpy는 한국어 형태소 분석에 사용할 수 있는 라이브러리입니다.
!pip install konlpy

# gradio는 텍스트 시각화 기능을 웹 UI로 만들 때 사용합니다.
!pip install gradio


## 17.2 예제 텍스트 준비

텍스트 시각화는 분석할 텍스트가 있어야 시작할 수 있습니다. 여기서는 LLM 교재의 주요 주제를 담은 예제 텍스트를 사용합니다.

실제 프로젝트에서는 다음 데이터를 사용할 수 있습니다.

- LLM 답변 기록
- RAG 검색 문서 내용
- PDF에서 추출한 텍스트
- Gradio 챗봇 대화 기록
- DB 챗봇이 생성한 자연어 답변


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 예제 텍스트
# 이 셀은 워드클라우드와 단어 빈도 분석에 사용할 예제 텍스트를 준비합니다.

# sample_text 변수에는 LLM 교재 주제를 담은 여러 문장을 저장합니다.
sample_text = """
LLM은 대규모 언어 모델을 의미합니다.
Transformer는 LLM의 핵심 구조이며 Attention 메커니즘을 사용합니다.
Hugging Face pipeline은 텍스트 분류, 요약, 번역, 이미지 분류를 쉽게 실행할 수 있게 합니다.
LangChain은 프롬프트, 모델, 파서, retriever, chain을 연결해 LLM 애플리케이션을 만듭니다.
RAG는 검색증강생성으로, 외부 문서를 검색한 뒤 LLM이 근거 기반 답변을 생성합니다.
FAISS와 Chroma는 벡터스토어로 사용할 수 있습니다.
Gradio는 LLM 챗봇 UI를 빠르게 만들 때 유용합니다.
LangGraph는 상태 기반 그래프로 복잡한 LLM 실행 흐름을 관리합니다.
LangSmith는 LLM 실행 기록을 추적하고 평가하는 데 사용합니다.
DB 챗봇은 자연어 질문을 SQL로 변환하고 데이터베이스 결과를 자연어로 설명합니다.
"""

# 예제 텍스트를 출력해 분석 대상 내용을 확인합니다.
print(sample_text)


## 17.3 텍스트 전처리 기본

텍스트 시각화를 하기 전에는 불필요한 문자를 정리하는 전처리가 필요합니다.

기본 전처리에는 다음 작업이 포함됩니다.

| 작업 | 의미 |
|---|---|
| 소문자 변환 | 영어 단어의 대소문자 차이를 줄임 |
| 특수문자 제거 | 쉼표, 마침표, 괄호 같은 분석 불필요 문자 제거 |
| 공백 정리 | 여러 공백과 줄바꿈 정리 |
| 불용어 제거 | 조사, 접속사, 너무 흔한 단어 제거 |

한국어는 띄어쓰기만으로 단어를 나누면 조사와 어미가 섞일 수 있습니다. 그래서 정확한 분석이 필요할 때는 형태소 분석기를 사용하는 것이 좋습니다.


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 텍스트 전처리
# 이 셀은 정규표현식을 사용해 텍스트를 간단히 정리합니다.

# re 모듈은 정규표현식으로 문자열을 검색하거나 치환할 때 사용합니다.
import re

# clean_text 함수는 원본 텍스트에서 분석에 불필요한 문자를 줄입니다.
def clean_text(text: str) -> str:
    # 입력 텍스트가 None이면 빈 문자열을 반환합니다.
    if text is None:
        # 분석할 텍스트가 없으므로 빈 문자열을 반환합니다.
        return ""

    # 영어 대소문자를 통일하기 위해 텍스트를 소문자로 변환합니다.
    lowered_text = text.lower()

    # 한글, 영어, 숫자, 공백을 제외한 특수문자를 공백으로 바꿉니다.
    cleaned_text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", lowered_text)

    # 여러 개의 공백을 하나의 공백으로 정리합니다.
    normalized_text = re.sub(r"\s+", " ", cleaned_text)

    # 앞뒤 공백을 제거한 문자열을 반환합니다.
    return normalized_text.strip()


# sample_text를 전처리합니다.
cleaned_sample_text = clean_text(sample_text)

# 전처리된 텍스트를 출력합니다.
print(cleaned_sample_text)


## 17.4 단어 토큰화와 불용어 제거

토큰화는 텍스트를 분석 단위로 나누는 작업입니다. 가장 단순한 방법은 공백 기준으로 나누는 것입니다.

불용어는 분석에서 큰 의미가 없는 단어입니다. 예를 들어 `은`, `는`, `이`, `가`, `을`, `를`, `그리고`, `the`, `is` 같은 단어는 빈도는 높지만 핵심 주제를 나타내지 않을 수 있습니다.

실습에서는 간단한 불용어 목록을 직접 만들어 사용합니다.


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 토큰화와 불용어 제거
# 이 셀은 텍스트를 단어로 나누고 불용어를 제거합니다.

# stopwords 변수에는 분석에서 제외할 단어 목록을 집합으로 저장합니다.
stopwords = {
    # 한국어 조사와 일반 불용어입니다.
    "은", "는", "이", "가", "을", "를", "에", "의", "와", "과", "로", "으로", "그리고", "또는",
    # 실습 문장에 자주 나오지만 분석 가치가 낮은 표현입니다.
    "수", "있습니다", "사용합니다", "의미합니다", "쉽게", "뒤", "때",
    # 영어 불용어입니다.
    "the", "is", "a", "an", "and", "or", "to", "of", "in", "for",
}

# tokenize_text 함수는 전처리된 텍스트를 단어 리스트로 변환합니다.
def tokenize_text(text: str) -> list[str]:
    # clean_text 함수를 사용해 텍스트를 먼저 정리합니다.
    cleaned = clean_text(text)

    # split은 공백 기준으로 문자열을 단어 리스트로 나눕니다.
    raw_tokens = cleaned.split()

    # 불용어가 아니고 길이가 2 이상인 단어만 남깁니다.
    tokens = [token for token in raw_tokens if token not in stopwords and len(token) >= 2]

    # 정리된 토큰 리스트를 반환합니다.
    return tokens


# sample_text를 토큰 리스트로 변환합니다.
tokens = tokenize_text(sample_text)

# 토큰 리스트를 출력합니다.
print(tokens)


## 17.5 단어 빈도 계산

단어 빈도는 각 단어가 텍스트 안에서 몇 번 등장했는지 세는 값입니다. Python의 `Counter`를 사용하면 쉽게 계산할 수 있습니다.

단어 빈도는 다음 시각화의 기초가 됩니다.

- 상위 키워드 표
- 단어 빈도 막대그래프
- 워드클라우드


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 단어 빈도 계산
# 이 셀은 Counter로 단어 빈도를 계산합니다.

# Counter는 리스트 안의 값이 몇 번 등장했는지 세는 도구입니다.
from collections import Counter

# pandas는 빈도 결과를 표 형태로 정리하기 위해 사용합니다.
import pandas as pd

# word_counts 변수에는 각 단어의 등장 횟수를 저장합니다.
word_counts = Counter(tokens)

# most_common은 가장 자주 등장한 단어를 순서대로 반환합니다.
top_words = word_counts.most_common(20)

# DataFrame은 단어와 빈도를 표 형태로 보여주기 위해 사용합니다.
frequency_df = pd.DataFrame(top_words, columns=["word", "count"])

# 단어 빈도 표를 출력합니다.
frequency_df


## 17.6 막대그래프로 단어 빈도 시각화

워드클라우드는 직관적이지만 정확한 수치 비교에는 막대그래프가 더 좋습니다. 빈도 상위 단어를 막대그래프로 그리면 어떤 단어가 가장 많이 등장했는지 명확하게 볼 수 있습니다.

한국어 폰트가 설정되지 않으면 그래프에서 한글이 깨질 수 있습니다. Windows에서는 보통 `Malgun Gothic` 폰트를 사용할 수 있습니다.


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 단어 빈도 막대그래프
# 이 셀은 단어 빈도를 막대그래프로 시각화합니다.

# matplotlib.pyplot은 그래프를 그리는 데 사용하는 모듈입니다.
import matplotlib.pyplot as plt

# Windows 환경에서 한글 표시를 위해 맑은 고딕 폰트를 지정합니다.
plt.rcParams["font.family"] = "Malgun Gothic"

# 마이너스 기호가 깨지는 것을 방지합니다.
plt.rcParams["axes.unicode_minus"] = False

# 그래프 크기를 지정합니다.
plt.figure(figsize=(10, 6))

# barh는 가로 막대그래프를 그립니다.
plt.barh(frequency_df["word"], frequency_df["count"])

# y축 순서를 뒤집어 가장 빈도가 높은 단어가 위에 오도록 합니다.
plt.gca().invert_yaxis()

# 그래프 제목을 지정합니다.
plt.title("상위 단어 빈도")

# x축 라벨을 지정합니다.
plt.xlabel("빈도")

# y축 라벨을 지정합니다.
plt.ylabel("단어")

# 그래프 요소가 겹치지 않도록 레이아웃을 조정합니다.
plt.tight_layout()

# 그래프를 화면에 표시합니다.
plt.show()


## 17.7 워드클라우드 만들기

워드클라우드는 단어 빈도를 글자 크기로 표현합니다. 많이 등장한 단어일수록 크게 표시됩니다.

WordCloud는 문자열 전체를 입력받거나, 단어 빈도 딕셔너리를 입력받을 수 있습니다. 여기서는 이미 계산한 단어 빈도 딕셔너리를 사용합니다.

한국어를 표시하려면 한글 폰트 경로를 지정해야 합니다. Windows에서는 일반적으로 다음 경로의 맑은 고딕 폰트를 사용할 수 있습니다.

```text
C:/Windows/Fonts/malgun.ttf
```


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 워드클라우드 생성
# 이 셀은 단어 빈도를 바탕으로 워드클라우드를 생성합니다.

# WordCloud는 단어 빈도를 이미지 형태의 워드클라우드로 변환합니다.
from wordcloud import WordCloud

# font_path 변수에는 한글을 표시할 폰트 파일 경로를 지정합니다.
font_path = "C:/Windows/Fonts/malgun.ttf"

# WordCloud 객체를 생성하고 크기, 배경색, 폰트를 지정합니다.
wordcloud = WordCloud(
    # width는 워드클라우드 이미지의 가로 크기입니다.
    width=900,
    # height는 워드클라우드 이미지의 세로 크기입니다.
    height=500,
    # background_color는 배경색입니다.
    background_color="white",
    # font_path는 한글 표시를 위한 폰트 경로입니다.
    font_path=font_path,
)

# generate_from_frequencies는 단어 빈도 딕셔너리로 워드클라우드를 생성합니다.
wordcloud_image = wordcloud.generate_from_frequencies(word_counts)

# 출력할 그림 크기를 지정합니다.
plt.figure(figsize=(12, 7))

# imshow는 워드클라우드 이미지를 화면에 표시합니다.
plt.imshow(wordcloud_image, interpolation="bilinear")

# axis off는 이미지 주변의 x축과 y축 표시를 숨깁니다.
plt.axis("off")

# 그래프 제목을 지정합니다.
plt.title("LLM 교재 예제 텍스트 워드클라우드")

# 워드클라우드를 화면에 표시합니다.
plt.show()


## 17.8 워드클라우드 이미지 저장

워드클라우드는 수업 자료, 보고서, 발표 자료에 넣기 위해 이미지 파일로 저장할 수 있습니다.

`to_file()` 메서드를 사용하면 PNG 파일로 저장할 수 있습니다.


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 워드클라우드 저장
# 이 셀은 생성한 워드클라우드를 이미지 파일로 저장합니다.

# output_image_path 변수에는 저장할 워드클라우드 이미지 파일명을 지정합니다.
output_image_path = "llm_wordcloud.png"

# to_file 메서드는 워드클라우드 이미지를 파일로 저장합니다.
wordcloud_image.to_file(output_image_path)

# 저장된 파일 경로를 출력합니다.
print("저장된 워드클라우드 파일:", output_image_path)


## 17.9 한국어 형태소 분석 사용하기

한국어는 조사와 어미가 단어에 붙기 때문에 공백 기준 토큰화만으로는 정확한 키워드 추출이 어렵습니다. 더 나은 분석을 위해서는 형태소 분석기를 사용할 수 있습니다.

`konlpy`의 `Okt`는 명사 추출에 자주 사용됩니다. 워드클라우드는 보통 명사 중심으로 만들면 주제가 더 명확해집니다.

단, KoNLPy는 Java 환경 설정이 필요할 수 있습니다. 환경 설정이 어렵다면 공백 기반 토큰화부터 사용해도 됩니다.


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - 한국어 명사 추출 예시
# 이 셀은 KoNLPy Okt를 사용해 한국어 명사를 추출하는 예시입니다.

# Okt는 한국어 형태소 분석기 중 하나입니다.
from konlpy.tag import Okt

# okt 객체를 생성합니다.
okt = Okt()

# extract_korean_nouns 함수는 텍스트에서 한국어 명사를 추출합니다.
def extract_korean_nouns(text: str) -> list[str]:
    # 입력 텍스트가 없으면 빈 리스트를 반환합니다.
    if not text:
        # 분석할 텍스트가 없으므로 빈 리스트입니다.
        return []

    # okt.nouns는 텍스트에서 명사 후보를 추출합니다.
    nouns = okt.nouns(text)

    # 길이가 2 이상이고 불용어가 아닌 명사만 남깁니다.
    filtered_nouns = [noun for noun in nouns if len(noun) >= 2 and noun not in stopwords]

    # 정리된 명사 리스트를 반환합니다.
    return filtered_nouns


# sample_text에서 한국어 명사를 추출합니다.
korean_nouns = extract_korean_nouns(sample_text)

# 추출된 명사 리스트를 출력합니다.
print(korean_nouns)


## 17.10 LLM 답변 분석 예시

LLM 챗봇을 여러 번 실행하면 답변 기록이 쌓입니다. 이 답변들을 모아서 워드클라우드를 만들면 모델이 어떤 표현을 자주 사용하는지 볼 수 있습니다.

예를 들어 다음을 확인할 수 있습니다.

- 답변이 특정 키워드에 지나치게 치우쳐 있는가
- RAG 답변에서 근거, 문서, 검색 같은 단어가 충분히 등장하는가
- DB 챗봇 답변에서 SQL, 결과, 조회 같은 표현이 반복되는가
- 교육용 챗봇이 학습자에게 쉬운 설명을 제공하는가


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - LLM 답변 분석 예시
# 이 셀은 여러 LLM 답변을 하나의 텍스트로 합쳐 분석하는 예시입니다.

# llm_answers 변수에는 챗봇이 생성한 답변 예시를 리스트로 저장합니다.
llm_answers = [
    # 첫 번째 답변 예시입니다.
    "RAG는 문서를 검색한 뒤 검색 결과를 근거로 답변을 생성하는 방식입니다.",
    # 두 번째 답변 예시입니다.
    "LangChain은 프롬프트와 LLM, 출력 파서를 연결해 체인을 구성합니다.",
    # 세 번째 답변 예시입니다.
    "DB 챗봇은 자연어 질문을 SQL로 변환하고 조회 결과를 설명합니다.",
    # 네 번째 답변 예시입니다.
    "LangGraph는 상태와 노드를 사용해 복잡한 실행 흐름을 그래프로 관리합니다.",
]

# join은 답변 리스트를 하나의 긴 문자열로 합칩니다.
joined_answers = "\n".join(llm_answers)

# 합쳐진 답변 텍스트를 토큰화합니다.
answer_tokens = tokenize_text(joined_answers)

# 답변 토큰의 단어 빈도를 계산합니다.
answer_counts = Counter(answer_tokens)

# 가장 자주 등장한 단어 10개를 출력합니다.
print(answer_counts.most_common(10))


## 17.11 Gradio 텍스트 시각화 UI

Gradio를 사용하면 사용자가 텍스트를 입력하고 바로 단어 빈도와 워드클라우드를 확인하는 간단한 웹 UI를 만들 수 있습니다.

기본 흐름은 다음과 같습니다.

```text
텍스트 입력
  -> 전처리
  -> 토큰화
  -> 단어 빈도 계산
  -> 워드클라우드 이미지 생성
  -> 표와 이미지 출력
```


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - Gradio UI 함수
# 이 셀은 입력 텍스트에서 단어 빈도 표와 워드클라우드 이미지를 만드는 함수를 정의합니다.

# tempfile은 임시 이미지 파일을 만들 때 사용합니다.
import tempfile

# gradio를 gr이라는 별칭으로 불러옵니다.
import gradio as gr

# analyze_text_visual 함수는 입력 텍스트를 분석해 표와 이미지 경로를 반환합니다.
def analyze_text_visual(text: str) -> tuple[pd.DataFrame, str]:
    # 입력 텍스트를 토큰화합니다.
    visual_tokens = tokenize_text(text)

    # 토큰이 없으면 빈 표와 None 이미지를 반환합니다.
    if not visual_tokens:
        # 빈 DataFrame을 만듭니다.
        empty_df = pd.DataFrame(columns=["word", "count"])

        # 분석할 단어가 없으므로 이미지 경로는 None입니다.
        return empty_df, None

    # 단어 빈도를 계산합니다.
    visual_counts = Counter(visual_tokens)

    # 상위 20개 단어를 DataFrame으로 정리합니다.
    visual_df = pd.DataFrame(visual_counts.most_common(20), columns=["word", "count"])

    # WordCloud 객체를 생성합니다.
    visual_wordcloud = WordCloud(
        # 이미지 가로 크기입니다.
        width=900,
        # 이미지 세로 크기입니다.
        height=500,
        # 배경색입니다.
        background_color="white",
        # 한글 폰트 경로입니다.
        font_path=font_path,
    ).generate_from_frequencies(visual_counts)

    # 임시 PNG 파일을 생성합니다.
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")

    # 워드클라우드를 임시 파일 경로에 저장합니다.
    visual_wordcloud.to_file(temp_file.name)

    # 단어 빈도 표와 워드클라우드 이미지 경로를 반환합니다.
    return visual_df, temp_file.name


In [ ]:
# 교재 위치: 17장 워드클라우드와 텍스트 시각화 - Gradio 텍스트 시각화 UI
# 이 셀은 텍스트 입력을 받아 단어 빈도 표와 워드클라우드를 출력하는 UI를 만듭니다.

# Blocks는 여러 Gradio 컴포넌트를 배치할 수 있는 컨테이너입니다.
with gr.Blocks() as text_visual_demo:
    # Markdown은 화면 제목을 표시합니다.
    gr.Markdown("# 텍스트 시각화 실습")

    # Textbox는 분석할 텍스트를 입력받습니다.
    text_input = gr.Textbox(label="분석할 텍스트", lines=10, value=sample_text)

    # Button은 분석 실행 버튼입니다.
    analyze_button = gr.Button("시각화하기")

    # Dataframe은 단어 빈도 표를 표시합니다.
    frequency_output = gr.Dataframe(label="단어 빈도")

    # Image는 워드클라우드 이미지를 표시합니다.
    wordcloud_output = gr.Image(label="워드클라우드")

    # 버튼 클릭 시 analyze_text_visual 함수를 실행합니다.
    analyze_button.click(
        # fn에는 실행할 분석 함수를 지정합니다.
        fn=analyze_text_visual,
        # inputs에는 텍스트 입력 컴포넌트를 지정합니다.
        inputs=text_input,
        # outputs에는 표와 이미지 출력 컴포넌트를 지정합니다.
        outputs=[frequency_output, wordcloud_output],
    )

# launch를 실행하면 로컬 웹 브라우저에서 텍스트 시각화 UI를 확인할 수 있습니다.
# text_visual_demo.launch()


## 17.12 실습 시 주의할 점

텍스트 시각화는 직관적이지만 해석할 때 주의가 필요합니다.

- 많이 등장한 단어가 항상 가장 중요한 단어는 아닙니다.
- 불용어 목록을 어떻게 정하느냐에 따라 결과가 달라집니다.
- 한국어는 형태소 분석 품질이 결과에 큰 영향을 줍니다.
- 짧은 텍스트에서는 워드클라우드가 큰 의미를 주지 못할 수 있습니다.
- LLM 답변은 반복 표현이 많아 특정 단어가 과도하게 커질 수 있습니다.
- 개인정보나 민감 정보가 포함된 텍스트를 시각화할 때는 저장과 공유에 주의해야 합니다.

워드클라우드는 탐색용 도구로 사용하고, 중요한 판단에는 원문과 정량 분석을 함께 확인하는 것이 좋습니다.


## 17.13 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| 한글이 네모로 표시됨 | 한글 폰트 미설정 | `font_path`에 한글 폰트 경로 지정 |
| WordCloud import 오류 | 패키지 미설치 | `pip install wordcloud` 실행 |
| KoNLPy 오류 | Java 환경 또는 JPype 문제 | Java 설치 확인 또는 공백 토큰화 사용 |
| 빈 워드클라우드 생성 | 토큰이 없거나 불용어가 너무 많음 | 불용어 목록과 입력 텍스트 확인 |
| 그래프 한글 깨짐 | matplotlib 폰트 설정 문제 | `plt.rcParams["font.family"] = "Malgun Gothic"` 설정 |
| Gradio 이미지 출력 실패 | 임시 파일 경로 문제 | PNG 파일이 생성되었는지 확인 |


## 17.14 정리

이 장에서는 LLM 실습에서 생성되는 텍스트를 시각화하는 방법을 정리했습니다.

핵심 정리는 다음과 같습니다.

- 텍스트 시각화는 많은 문장 속 주요 단어와 흐름을 빠르게 파악하는 데 유용합니다.
- 전처리, 토큰화, 불용어 제거가 시각화 품질을 크게 좌우합니다.
- `Counter`로 단어 빈도를 계산할 수 있습니다.
- 막대그래프는 단어 빈도를 정확히 비교하는 데 좋습니다.
- 워드클라우드는 주요 키워드를 직관적으로 보여주는 데 좋습니다.
- 한국어 워드클라우드에는 한글 폰트 설정이 필요합니다.
- KoNLPy 같은 형태소 분석기를 사용하면 한국어 명사 중심 분석이 가능합니다.
- Gradio로 텍스트 입력과 워드클라우드 출력을 연결할 수 있습니다.

다음 장에서는 자기소개서 도우미 챗봇을 다룹니다. LLM을 사용해 사용자의 자기소개서 문장을 개선하고, 항목별 피드백을 제공하는 실습으로 확장합니다.
